In [4]:
import pandas as pd
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
from dotenv import load_dotenv

load_dotenv()

True

In [5]:

# =====================================================
# CONFIG
# =====================================================

INPUT_PARQUET = "../data/silver/1.enriched_tracks.parquet"
OUTPUT_PARQUET = "../data/silver/2.tracks_with_musicbrainz.parquet"

MAX_WORKERS = 5  # MusicBrainz requires low request rate
REQUEST_DELAY = 1.1  # required by MusicBrainz (VERY important)
BATCH_SIZE = 300
START_FROM = 0   # change this later to continue
HEADERS = {
    "User-Agent": "spotify-audio-project/1.0 (lihermann@outlook.com)"
}

In [ ]:

print("Loading silver dataset...")
df = pd.read_parquet(INPUT_PARQUET)

print(f"Tracks to enrich: {len(df)}")

if os.path.exists(OUTPUT_PARQUET):
    done_df = pd.read_parquet(OUTPUT_PARQUET)
    print(f"Already processed: {len(done_df)} tracks")

    # remove already processed tracks
    df = df[~df["spotify_track_uri"].isin(done_df["track_id"])]

    print(f"Remaining tracks: {len(df)}")
else:
    done_df = pd.DataFrame()

batch = df.head(BATCH_SIZE)

print(f"Processing {len(batch)} tracks in this run...\n")

results = []

Loading silver dataset...
Tracks to enrich: 51606
Already processed: 440 tracks


KeyError: 'spotify_track_uri'

In [ ]:
def search_musicbrainz(track, artist):

    try:
        url = "https://musicbrainz.org/ws/2/recording/"

        query = f'recording:"{track}" AND artist:"{artist}"'

        params = {
            "query": query,
            "fmt": "json",
            "limit": 1
        }

        response = requests.get(url, params=params)

        if response.status_code != 200:
            return None

        data = response.json()

        if "recordings" not in data:
            return None

        if len(data["recordings"]) == 0:
            return None

        rec = data["recordings"][0]

        return {
            "mbid": rec.get("id"),
            "title_mb": rec.get("title"),
            "artist_mb": rec["artist-credit"][0]["name"]
        }

    except Exception:
        return None

In [ ]:
for i, row in batch.iterrows():

    track = row["master_metadata_track_name"]
    artist = row["master_metadata_album_artist_name"]

    print(f"Searching: {track} - {artist}")

    mb_data = search_musicbrainz(track, artist)

    if mb_data is None:
        mbid = None
        title_mb = None
        artist_mb = None
    else:
        mbid = mb_data["mbid"]
        title_mb = mb_data["title_mb"]
        artist_mb = mb_data["artist_mb"]
    result = {
        "track_id": row["track_id"],
        "track": track,
        "artist": artist,
        "mbid": mbid,
        "title_mb": title_mb,
        "artist_mb": artist_mb
    }

    results.append(result)

    # save after every track (no data loss if it crashes)
    temp_df = pd.DataFrame(results)
    full_df = pd.concat([done_df, temp_df], ignore_index=True)
    full_df.to_parquet(OUTPUT_PARQUET, index=False)

    time.sleep(REQUEST_DELAY)

print("\nBatch finished.")
print("Run the script again to continue automatically.")

Searching: Nightwood - Two Steps from Hell
Searching: Still Got Time (feat. PARTYNEXTDOOR) - ZAYN
Searching: Welcome to Boston - Lorne Balfe
Searching: The Distance - Poets of the Fall
Searching: Mr. Brightside - The Killers
Searching: Too Many Friends - Placebo
Searching: Guide Vocal - 2007 Remaster - Genesis
Searching: Unconditionally - Katy Perry
Searching: Devils Island - 2004 Remaster - Megadeth
Searching: Franco Un-American - NOFX
Searching: Mayor Que Yo 3 - Luny Tunes
Searching: Memories In Shadow - Magic Sword
Searching: One Right Now (with The Weeknd) - Post Malone
Searching: Telephone - Lady Gaga
Searching: Someday I'll Be Saturday Night - Bon Jovi
Searching: Intro - dialogue - Kansas
Searching: Something Different - Godsmack
Searching: Bossa Nova Party - Cafe Music BGM channel
Searching: Violet Chemistry - Miley Cyrus
Searching: Ever Closer - In the Silence
Searching: Scavenger - Calyx & TeeBee
Searching: IN MY REMAINS - Linkin Park
Searching: sincerely - Nessa Barrett
Searc

In [ ]:
BATCH_SIZE = 300
COOLDOWN = 60   # pause 1 minute between batches

while True:

    print("\nStarting new batch...\n")

    batch = df.head(BATCH_SIZE)

    if len(batch) == 0:
        print("All tracks processed.")
        break

    for i, row in batch.iterrows():

        track = row["master_metadata_track_name"]
        artist = row["master_metadata_album_artist_name"]

        mb_data = search_musicbrainz(track, artist)

        result = {
            "track_id": row["track_id"],
            "track": track,
            "artist": artist,
            "mbid": None if mb_data is None else mb_data["mbid"]
        }

        results.append(result)

        # save immediately
        temp_df = pd.DataFrame(results)
        full_df = pd.concat([done_df, temp_df], ignore_index=True)
        full_df.to_parquet(OUTPUT_PARQUET, index=False)

        time.sleep(1.1)   # MusicBrainz rate limit

    print("Batch finished. Cooling down 60 seconds...")
    time.sleep(COOLDOWN)


Starting new batch...

